# 🩺 Huấn luyện Mamba-UNet (MambaVision Backbone) trên Google Colab

Quy trình toàn diện: Clone mã nguồn từ GitHub -> Cài đặt thư viện -> Mount Drive -> Giải nén chọn số lượng ca bệnh nhân vào SSD -> Huấn luyện -> Đánh giá 3D -> Lưu Checkpoint về Drive.

In [ ]:
# 1. Kiểm tra phần cứng và GPU khả dụng
!nvidia-smi

In [ ]:
# 2. Tải và đồng bộ mã nguồn từ GitHub
import os
import sys

GITHUB_REPO = 'https://github.com/NguyenMinhTri24072005/CT208E_LIDC_DuneAI_MambaVision.git'
BRANCH = 'main'  # Nhánh mặc định trên GitHub (đổi nếu bạn dùng nhánh khác)
REPO_DIR = '/content/CT208E_LIDC_DuneAI_MambaVision'

if os.path.exists('mamba_unet') and os.path.exists('common'):
    print(f'✅ Đang chạy trực tiếp trong thư mục dự án: {os.getcwd()}')
else:
    if not os.path.exists(REPO_DIR):
        print(f'📥 Đang clone repository từ GitHub (nhánh {BRANCH})...')
        !git clone -b {BRANCH} {GITHUB_REPO} {REPO_DIR}
    else:
        print(f'🔄 Đang đồng bộ mã nguồn mới nhất từ GitHub...')
        %cd {REPO_DIR}
        !git fetch origin
        !git checkout {BRANCH}
        !git pull origin {BRANCH}
    %cd {REPO_DIR}

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'🚀 Thư mục làm việc hiện tại: {os.getcwd()}')

In [ ]:
# 3. Cài đặt các thư viện phụ thuộc
!pip install -q SimpleITK pandas scipy opencv-python-headless pyyaml matplotlib

In [ ]:
# 3b. (Tùy chọn) Cài đặt nhân CUDA mamba-ssm
# Lưu ý: Mamba-UNet hỗ trợ tự động Fallback thuần PyTorch nếu không cài được mamba-ssm
import torch
if torch.cuda.is_available():
    try:
        print('⚡ Thử cài đặt causal-conv1d và mamba-ssm để tăng tốc GPU...')
        !pip install -q causal-conv1d mamba-ssm
        print('✅ Cài đặt mamba-ssm thành công!')
    except Exception as e:
        print('⚠️ Sẽ sử dụng PyTorch selective_scan fallback mặc định:', e)

In [ ]:
# 4. Kết nối với Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 5. Chuẩn bị dữ liệu từ Google Drive vào SSD cục bộ (/content/data/cache_1mm)
# Thư mục GG_1 trên máy tính được mirror trực tiếp vào Google Drive.
# Mã sẽ tự động tìm kiếm file zip theo đúng cấu trúc mirror của bạn:
import os
from common.data_pipeline.runtime_data import run_colab

DRIVE_ZIP_CANDIDATES = [
    # 1. Mirror trực tiếp thư mục GG_1 vào MyDrive:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 2. Mirror có thư mục cha GG_1 trong MyDrive:
    '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/cache_1mm.zip',
    # 3. File zip tại thư mục KAGGLE_UPLOAD:
    '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/KAGGLE_UPLOAD/cache_1mm.zip',
    # 4. File zip đặt ngay tại gốc MyDrive:
    '/content/drive/MyDrive/cache_1mm.zip',
]

DRIVE_ZIP_PATH = None
for cand in DRIVE_ZIP_CANDIDATES:
    if os.path.exists(cand):
        DRIVE_ZIP_PATH = cand
        print(f'🎯 Đã tìm thấy file zip: {DRIVE_ZIP_PATH}')
        break

if DRIVE_ZIP_PATH is None:
    print('⚠️ Chưa thấy ở các đường dẫn cố định, sẽ tự động dò tìm sâu dưới /content/drive...')
    DRIVE_ZIP_PATH = 'auto'

# NUM_PATIENTS: 0 = giải nén toàn bộ 1010 ca (cần ~40-60GB SSD Colab),
# hoặc chọn số lượng ca (vd: 50, 100, 200) để huấn luyện nhanh tránh tràn ổ đĩa Colab
NUM_PATIENTS = 0

cache_dir = run_colab(
    zip_path=DRIVE_ZIP_PATH,
    dest_dir='/content/data/cache_1mm',
    n_patients=NUM_PATIENTS,
)
print(f'✅ Dữ liệu sẵn sàng tại: {cache_dir}')

In [ ]:
# 6. Bắt đầu quá trình huấn luyện
!python -m mamba_unet.train_main train \
    --cache-dir /content/data/cache_1mm \
    --backbone mamba_vision_T \
    --pretrained \
    --freeze-backbone-epochs 5 \
    --backbone-lr-mult 0.1 \
    --epochs 50 \
    --batch-size 8 \
    --lr 1e-4 \
    --tag colab_run

In [ ]:
# 7. Đánh giá 3D cấp độ bệnh nhân trên tập Validation
!python -m mamba_unet.evaluate eval \
    --cache-dir /content/data/cache_1mm \
    --checkpoint checkpoints/best_model.pth \
    --split val

In [ ]:
# 8. Sao lưu kết quả và checkpoint về Google Drive (tự động đồng bộ về máy tính qua mirror GG_1)
import os

# Đường dẫn lưu checkpoint trực tiếp vào thư mục mirror GG_1 trên Drive để máy tính tự động đồng bộ về:
MIRROR_SOURCES_DIR = '/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/GG_1/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES'
if not os.path.exists(MIRROR_SOURCES_DIR):
    MIRROR_SOURCES_DIR = '/content/drive/MyDrive/DuneAI_Backups'

BACKUP_RUNS = os.path.join(MIRROR_SOURCES_DIR, 'mamba_unet/runs')
BACKUP_CKPT = os.path.join(MIRROR_SOURCES_DIR, 'checkpoints')
!mkdir -p '{BACKUP_RUNS}' '{BACKUP_CKPT}'
!cp -ru mamba_unet/runs/* '{BACKUP_RUNS}/' 2>/dev/null || true
!cp -ru checkpoints/* '{BACKUP_CKPT}/' 2>/dev/null || true
print(f'✅ Đã sao lưu checkpoints và logs vào Google Drive tại: {MIRROR_SOURCES_DIR}')